## Transformer-based NLI Classifier

Ce notebook implémente un modèle de base d'inférence en langage naturel (NLI) basé sur l'encodeur Transformer, utilisant PyTorch. Il comprend un mécanisme d'attention à une seule tête, un mécanisme d'attention à plusieurs têtes, des réseaux de neurones à propagation avant et une pile complète d'encodeurs Transformer. Le modèle sera entraîné sur le fichier train.csv et évalué sur le fichier test.csv ; les prédictions seront enregistrées dans le fichier sample_submission.csv.

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence
import einops # Optional, but very useful for reshaping
import numpy as np

# Define hyperparameters
hidden_dim = 64
num_heads = 8
ff_dim = 128
num_layers = 2
dropout_rate = 0.1
max_seq_len = 50 # Increased max sequence length for potentially longer texts
batch_size = 32 # Increased batch size
num_epochs = 10 # Increased epochs
lr = 1e-4

### 1. Single-Head Attention Implementation

In [18]:
class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        q = self.query_proj(x)
        k = self.key_proj(x)
        v = self.value_proj(x)

        attention_scores = torch.matmul(q, k.transpose(-2, -1))
        attention_scores = attention_scores / (self.hidden_dim ** 0.5)
        attention_weights = F.softmax(attention_scores, dim=-1)

        output = torch.matmul(attention_weights, v)

        return output, attention_weights

### 2. Multi-Head Attention Module

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.qkv_proj = nn.Linear(hidden_dim, hidden_dim * 3)
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout_rate)
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        norm_x = self.layer_norm(x)
        batch_size, seq_len, hidden_dim = norm_x.shape
        qkv = self.qkv_proj(norm_x)

        q, k, v = einops.rearrange(qkv, 'b s (three h d) -> three b h s d', h=self.num_heads, d=self.head_dim, three=3)

        attention_scores = torch.matmul(q, k.transpose(-2, -1))
        attention_scores = attention_scores / (self.head_dim ** 0.5)
        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        attn_output = torch.matmul(attention_weights, v)
        attn_output = einops.rearrange(attn_output, 'b h s d -> b s (h d)')

        output = self.output_proj(attn_output)
        output = self.dropout(output)
        output = output + x

        return output, attention_weights

### 3. Transformer Encoder Stack

In [20]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, hidden_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.linear1 = nn.Linear(hidden_dim, ff_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear2 = nn.Linear(ff_dim, hidden_dim)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class TransformerEncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
        self.feed_forward = FeedForwardNetwork(hidden_dim, ff_dim, dropout_rate)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        attn_output, _ = self.multi_head_attention(x)
        norm_attn_output = self.norm1(attn_output)
        ff_output = self.feed_forward(norm_attn_output)
        ff_output = self.dropout2(ff_output)
        output = ff_output + attn_output

        return output

class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, hidden_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(hidden_dim, num_heads, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.final_norm(x)

### 4. NLI Classifier with Embedding Layer

In [21]:
class NLIClassifierWithEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, transformer_encoder, hidden_dim, num_classes, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # <pad> is 0
        self.positional_encoding = nn.Parameter(torch.randn(1, max_seq_len * 2, embedding_dim)) # For combined sequence
        self.encoder = transformer_encoder
        assert hidden_dim == embedding_dim, "hidden_dim of the encoder must be equal to embedding_dim"
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        embedded_input = self.embedding(input_ids)

        # Add positional encoding (simplified for now)
        # In a real transformer, positional encoding is typically added before the first encoder block.
        # Here, we're adding it after embedding and assuming a fixed max_seq_len for the combined input.
        if embedded_input.size(1) > self.positional_encoding.size(1):
            raise ValueError(f"Input sequence length ({embedded_input.size(1)}) exceeds maximum positional encoding length ({self.positional_encoding.size(1)}).")

        embedded_input = embedded_input + self.positional_encoding[:, :embedded_input.size(1), :]

        encoded_output = self.encoder(embedded_input)
        pooled_output = encoded_output.mean(dim=1)
        logits = self.classifier(pooled_output)
        return logits

### 5. Data Loading and Preprocessing

In [22]:
# Load train and test data
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Handle potential missing values in text columns by converting to string
train_df['premise'] = train_df['premise'].astype(str)
train_df['hypothesis'] = train_df['hypothesis'].astype(str)
test_df['premise'] = test_df['premise'].astype(str)
test_df['hypothesis'] = test_df['hypothesis'].astype(str)

# Map labels to numerical IDs for training data
if 'label' in train_df.columns:
    unique_labels = train_df['label'].unique()
    label_to_id = {label: i for i, label in enumerate(unique_labels)}
    train_df['label_id'] = train_df['label'].map(label_to_id)
    num_classes = len(unique_labels)
else:
    print("Warning: 'label' column not found in train.csv. Using dummy labels.")
    train_df['label_id'] = 0
    num_classes = 1 # Fallback

print(f"Number of classes: {num_classes}")
print(f"Label to ID mapping: {label_to_id}")

# Simple tokenizer and vocabulary builder
vocab = defaultdict(lambda: len(vocab)) # Use 0 for padding
vocab['<pad>'] = 0
vocab['<unk>'] = 1 # For unknown words

def tokenize_and_to_ids(text, vocab_dict, max_len):
    tokens = text.lower().split()
    ids = [vocab_dict[token] if token in vocab_dict else vocab_dict['<unk>'] for token in tokens]
    return ids[:max_len]

# Process training data
train_premise_ids = [tokenize_and_to_ids(p, vocab, max_seq_len) for p in train_df['premise']]
train_hypothesis_ids = [tokenize_and_to_ids(h, vocab, max_seq_len) for h in train_df['hypothesis']]

# Convert current vocab to a fixed dictionary for test data tokenization
fixed_vocab = dict(vocab)
vocab_size = len(fixed_vocab)
print(f"Vocabulary size (training data): {vocab_size}")

# Process test data using the vocabulary learned from training data
test_premise_ids = [tokenize_and_to_ids(p, fixed_vocab, max_seq_len) for p in test_df['premise']]
test_hypothesis_ids = [tokenize_and_to_ids(h, fixed_vocab, max_seq_len) for h in test_df['hypothesis']]

# Padding function
def pad_and_combine(premise_ids_list, hypothesis_ids_list, padding_value=0):
    premises_padded = pad_sequence([torch.tensor(ids, dtype=torch.long) for ids in premise_ids_list], batch_first=True, padding_value=padding_value)
    hypotheses_padded = pad_sequence([torch.tensor(ids, dtype=torch.long) for ids in hypothesis_ids_list], batch_first=True, padding_value=padding_value)

    # Combine premise and hypothesis
    combined_input = torch.cat((premises_padded, hypotheses_padded), dim=1)
    return combined_input

train_combined_input_ids = pad_and_combine(train_premise_ids, train_hypothesis_ids)
test_combined_input_ids = pad_and_combine(test_premise_ids, test_hypothesis_ids)

train_labels_tensor = torch.tensor(train_df['label_id'].tolist(), dtype=torch.long)

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(train_combined_input_ids, train_labels_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_combined_input_ids)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train combined input IDs shape: {train_combined_input_ids.shape}")
print(f"Test combined input IDs shape: {test_combined_input_ids.shape}")
print(f"Train labels shape: {train_labels_tensor.shape}")

Number of classes: 3
Label to ID mapping: {np.int64(0): 0, np.int64(2): 1, np.int64(1): 2}
Vocabulary size (training data): 2
Train combined input IDs shape: torch.Size([12120, 96])
Test combined input IDs shape: torch.Size([5195, 85])
Train labels shape: torch.Size([12120])


### 6. Model Initialization

In [23]:
embedding_dim = hidden_dim # Must match for encoder

transformer_encoder_instance = TransformerEncoder(num_layers, hidden_dim, num_heads, ff_dim, dropout_rate)
model = NLIClassifierWithEmbedding(vocab_size, embedding_dim, transformer_encoder_instance, hidden_dim, num_classes, max_seq_len)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

Using device: cpu


### 7. Training Loop

In [24]:
print(f"\n--- Starting Training Loop for {num_epochs} epochs ---")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (data, labels) in enumerate(train_dataloader):
        data, labels = data.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_dataloader)
    accuracy = 100 * correct_predictions / total_samples
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("--- Training Loop Finished ---")


--- Starting Training Loop for 10 epochs ---
Epoch 1/10, Loss: 1.0991, Accuracy: 33.68%
Epoch 2/10, Loss: 1.0987, Accuracy: 34.32%
Epoch 3/10, Loss: 1.0981, Accuracy: 34.32%
Epoch 4/10, Loss: 1.0972, Accuracy: 34.74%
Epoch 5/10, Loss: 1.0951, Accuracy: 35.40%
Epoch 6/10, Loss: 1.0929, Accuracy: 36.35%
Epoch 7/10, Loss: 1.0928, Accuracy: 35.97%
Epoch 8/10, Loss: 1.0924, Accuracy: 36.30%
Epoch 9/10, Loss: 1.0917, Accuracy: 36.76%
Epoch 10/10, Loss: 1.0914, Accuracy: 36.91%
--- Training Loop Finished ---


### 8. Evaluation on Test Data and Submission File Generation

In [25]:
print("\n--- Evaluating on Test Data and Generating Submission File ---")

model.eval() # Set model to evaluation mode
predictions = []

with torch.no_grad():
    for data in test_dataloader:
        data = data[0].to(device) # Dataloader for test_dataset yields a tuple (data,) or similar
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        predictions.extend(predicted.cpu().numpy())

# Convert numerical predictions back to original labels if mapping was used
id_to_label = {v: k for k, v in label_to_id.items()}
predicted_labels = [id_to_label[p] for p in predictions]

# Create submission DataFrame
submission_df = pd.DataFrame({'id': test_df['id'], 'prediction': predicted_labels})

# Save to sample_submission.csv
submission_df.to_csv('sample_submission.csv', index=False)
print("Submission file 'sample_submission.csv' generated successfully.")
print("First 5 rows of submission file:")
display(submission_df.head())


--- Evaluating on Test Data and Generating Submission File ---
Submission file 'sample_submission.csv' generated successfully.
First 5 rows of submission file:


,id,prediction
0,c6d58c3f69,0
1,cefcc82292,1
2,e98005252c,0
3,58518c10ba,0
4,c32b0d16df,0
